> **Open in VS Code:** From your cloned `s26-06643` repository, run in the terminal:
> ```bash
> code sse/optional/case-studies/case-studies.ipynb
> ```
> [View source on GitHub](https://github.com/jkitchin/s26-06643/blob/main/sse/optional/case-studies/case-studies.ipynb)

# Case Studies in AI-Assisted Software Development

The best way to understand what AI coding agents can and cannot do is to examine real projects — not toy demos, but real software built by real people (and AI) for real purposes. This module examines five case studies that span the spectrum from remarkable success to instructive failure.

Each case study tells us something different about the current state of AI-assisted development:

| Case Study | What It Shows |
|-----------|---------------|
| JustHTML port | What works when AI has a comprehensive test suite |
| Claude's C Compiler | What 16 parallel AI agents can build — and what they can't |
| Cursor's FastRender browser | What happens when AI produces code at scale without quality control |
| LLM SQLite rewrite | Why plausible code is not correct code — and why benchmarks matter |
| scimax for VS Code | What a domain expert can build with AI in 43 days |

## Case Study 1: Porting JustHTML — 9,000 Lines in 4.5 Hours

### The Project

In December 2025, [Simon Willison](https://simonwillison.net/2025/Dec/15/porting-justhtml/) ported **JustHTML**, an HTML5 parsing library, from Python to JavaScript using Codex CLI with GPT-5.2. The result: a working library that passes 9,200 tests from the html5lib-tests suite.

### The Numbers

| Metric | Value |
|--------|-------|
| Time elapsed | 4.5 hours |
| Lines of code | ~9,000 across 43 commits |
| Test suite | 9,200 tests (html5lib-tests) |
| Tests passing | 8,810/8,822 (99.9%) |
| Human prompts | ~8 initial prompts, then "Keep going" |
| Input tokens | 1.46M (plus 97M cached) |
| Output tokens | 626K |

### How It Worked

Willison's approach had three key elements:

1. **A specification document.** He examined the original Python implementation and created a specification describing what the JavaScript version should do.

2. **An existing, comprehensive test suite.** The html5lib-tests suite (9,200 tests) provided an external, pre-existing validation framework. The AI could run tests after each change and verify its work independently.

3. **Minimal human intervention.** After establishing the initial setup (~8 prompts), Willison used `--yolo` mode (auto-approve all actions) and let the AI work autonomously. His primary intervention was "Keep going" after a token limit reset.

The model worked uninterrupted for several hours, committing and pushing frequently. Willison was decorating for Christmas and watching a film during most of the process.

### Why It Worked

This case study succeeds because of a specific combination:

- **Well-defined translation task.** Porting from one language to another is structurally bounded — the *what* is already decided, only the *how* changes.
- **Comprehensive external test suite.** The AI had a reliable signal for correctness at every step. It could iterate autonomously because it knew when it was wrong.
- **No novel design decisions.** The architecture, algorithms, and edge cases were already solved in the source implementation.

### What It Teaches

AI excels at **translation with verification**. When you have (1) a clear specification, (2) a comprehensive test suite, and (3) a bounded scope, AI agents can work with minimal supervision and produce high-quality results. This aligns with the plan-execute workflow from L00 — the plan was the specification, and the tests provided continuous feedback.

**The critical factor was not AI capability — it was the test suite.** Without those 9,200 tests, there would be no way to know if the port was correct. Willison himself emphasized this: the test suite is what made autonomous operation possible.

## Case Study 2: Claude's C Compiler — 100,000 Lines, $20,000, Two Weeks

### The Project

In February 2026, [Nicholas Carlini at Anthropic](https://www.anthropic.com/engineering/building-c-compiler) orchestrated **16 parallel Claude instances** (Opus 4.6) to build a C compiler in Rust from scratch. The compiler, called CCC (Claude's C Compiler), can compile the Linux kernel, QEMU, FFmpeg, SQLite, PostgreSQL, and Redis.

### The Numbers

| Metric | Value |
|--------|-------|
| Duration | ~2 weeks |
| Lines of code | ~100,000 (Rust) |
| Claude Code sessions | 2,000 |
| Input tokens | 2 billion |
| Output tokens | 140 million |
| API cost | ~$20,000 |
| Test pass rate | 99% (GCC torture test suite) |
| Target architectures | x86 (64/32-bit), ARM, RISC-V |

### How It Worked

The system used a novel multi-agent coordination scheme:

- Each agent ran in a **Docker container** with access to a shared git repository
- Agents **claimed tasks** by writing lock files to prevent duplicate work
- Agents pulled, made changes, resolved merge conflicts, and pushed
- When an agent finished, a fresh one spawned to continue
- Carlini assigned **specialized roles**: some agents focused on optimization, others on code quality, others on documentation

A key technique was using **GCC as an oracle**: the system randomly mixed Claude-compiled and GCC-compiled kernel files to isolate which Claude-compiled files caused failures.

### What Worked

- **Comprehensive test suites** (GCC torture tests) gave agents a reliable correctness signal
- **Task decomposition** into independent, parallelizable units
- **Deterministic subsampling** (`--fast` option) let agents catch regressions without running full suites
- **Oracle comparison** (mixing with GCC output) for debugging

### What Didn't Work

- **Single monolithic tasks** caused all agents to hit the same bug and overwrite each other's fixes
- **Time blindness**: agents spent hours on unnecessary tests without explicit progress tracking
- **Context window pollution**: unstructured error output consumed context, requiring machine-readable logging
- **Feature interference**: implementing new features frequently broke existing ones

### Community Reaction

The project received significant scrutiny:

- **Chris Lattner** (creator of Swift and LLVM) [called it](https://simonwillison.net/2026/Feb/22/ccc/) "a competent textbook implementation" comparable to what a strong undergraduate team might produce — praise with important caveats about production readiness.
- **The Register** [noted](https://www.theregister.com/2026/02/13/anthropic_c_compiler/) it could not compile "Hello World" without manually specifying library paths — a basic usability failure.
- **Community testers** found it lacked a standalone assembler/linker (it uses GCC's) and produced significantly less efficient code than GCC even with optimizations.
- Lattner's key observation: the compiler showed **"optimization toward passing tests rather than building general abstractions"** — it solved the test suite rather than building a general-purpose tool.

### What It Teaches

The CCC project reveals both the power and limits of multi-agent AI systems:

1. **Scale is achievable.** 100,000 lines of working code is genuinely impressive.
2. **Test-driven development works for AI too.** The test suite was the essential ingredient, just as in the JustHTML case.
3. **Passing tests is not the same as building good software.** The compiler optimized for test passage rather than generality — a form of overfitting.
4. **Infrastructure matters as much as intelligence.** Carlini's contribution was designing the coordination system, not writing code. The human role shifted from implementation to orchestration.
5. **$20,000 and two weeks for a "textbook" compiler** raises questions about cost-effectiveness. Computer science students build compilers as course projects.

## Case Study 3: Cursor's FastRender Browser — 3 Million Lines of AI Slop

### The Project

In January 2026, [Cursor CEO Michael Truell announced](https://x.com/mntruell/status/2011562190286045552) that hundreds of GPT-5.2 agents had built a web browser called **FastRender** in one week. The claims were dramatic: 3 million+ lines of Rust code, a rendering engine built "from scratch" with HTML parsing, CSS cascade, layout, text shaping, and a "custom JavaScript VM." Approximately 2,000 concurrent agents ran at peak, consuming an estimated 3 billion tokens.

Truell's own assessment was candid: *"It \*kind of\* works!"* Simple websites rendered "quickly and largely correctly."

### What the Community Found

When developers examined the repository, the reality was starkly different from the narrative:

**Compilation failures:**
Running `cargo build` immediately produced errors — 34 compilation errors and 94 warnings. All GitHub Actions CI runs on the main branch had failed.

**Not actually "from scratch":**
The claimed "custom JavaScript VM" and "from-scratch rendering engine" were wrappers around existing libraries: `html5ever` (Mozilla's HTML parser), `cssparser`, and `rquickjs` (a Rust binding to QuickJS). Community members described it as essentially a wrapper around [Servo](https://servo.org/) components.

**Massive bloat:**
Developer Oliver Medhurst pointed out that competing browser engines like Ladybird and Servo "do much more in much less lines of code (~1M)." The 3 million lines was a sign of duplication and waste, not capability.

**Professional code quality analysis:**
The [Software Improvement Group (SIG)](https://www.softwareimprovementgroup.com/blog/quality-of-fastrender/) — a firm that benchmarks code quality against a database of 400+ billion lines across 300+ technologies — analyzed FastRender and found:

| Metric | FastRender Score | Industry Benchmark |
|--------|-----------------|-------------------|
| Maintainability | 1.3 / 5 stars | Bottom 5% of all systems |
| Architecture quality | 2.1 / 5 stars | Below typical market |

SIG concluded that components were "tightly interwoven," changes would likely create unexpected side effects, and despite being newly generated, the code exhibited patterns typical of legacy systems. They estimated that changes to a comparable four-star codebase take roughly four times less effort.

**88% CI failure rate:**
Jason Gorman of Codemanship highlighted that the project's CI/CD metrics showed an 88% job failure rate, characterizing it as proof "that agentic AI scales for creating broken software."

### What It Teaches

FastRender is the anti-JustHTML. It shows what happens when AI operates at scale *without* the conditions that make AI-assisted development successful:

| Factor | JustHTML (success) | FastRender (failure) |
|--------|--------------------|---------------------|
| Test suite | 9,200 external tests | Failing CI |
| Scope | Bounded (port a library) | Unbounded (build a browser) |
| Verification | Continuous, automated | Minimal |
| Human oversight | Light but targeted | Minimal |
| Outcome | 99.9% tests passing | Doesn't compile |

Key lessons:

1. **Lines of code is not a metric of success.** 3 million lines that don't compile is worse than zero lines. SIG estimated it at 110 person-years of effort — but the result scored in the bottom 5% for maintainability.
2. **"From scratch" claims need verification.** AI often wraps existing libraries rather than reimplementing them — and may not disclose this.
3. **Scale without quality control produces waste.** More agents producing more code faster just creates more broken code faster.
4. **The test suite is the ceiling.** Without tests, there is no way for agents to know when they are wrong, so they cannot self-correct.
5. **Be skeptical of demos.** "Simple websites render largely correctly" is a long way from a working browser. Always look at the repo.

## Case Study 4: LLM SQLite Rewrite — 576,000 Lines of Plausible Rust

### The Project

In early 2026, a developer known as [Hōrōshi (KatanaQuant)](https://katanaquant.com) used an LLM to rewrite SQLite from scratch in Rust. The result was 576,000 lines of code that compiled, ran, and appeared to work — until benchmarking revealed it was **20,171 times slower** than the original SQLite on primary key lookups.

The analysis, published as "[Your LLM Doesn't Write Correct Code. It Writes Plausible Code](https://katanaquant.com/blog/llm-plausible-code)," became one of the most widely cited critiques of AI-generated code in 2026.

### The Numbers

| Metric | Value |
|--------|-------|
| Lines of code | 576,000 (Rust) |
| Language | Rust (rewriting C) |
| Benchmark: primary key lookup | 20,171x slower than SQLite |
| Root causes found | 2 critical bugs + multiple compound inefficiencies |

### The Bugs

**Bug #1: Missing integer primary key (iPK) check.**

SQLite uses a well-known optimization: when a table has an integer primary key, the rowid *is* the primary key, so lookups go directly to the B-tree. The LLM's query planner recognized named columns but never checked whether the column was an iPK alias. Result: every primary key lookup triggered a full table scan instead of a direct B-tree seek.

This is a textbook optimization that any database engineer would know — but the LLM produced structurally correct code that simply omitted the critical check. The code *looked* right. The query planner *worked*. It just never took the fast path.

**Bug #2: `fsync` on every statement instead of `fdatasync`.**

The LLM used `fsync()` after every SQL statement. SQLite uses `fdatasync()` and batches syncs within transactions. The difference: `fsync` flushes all file metadata (permissions, timestamps, size); `fdatasync` only flushes data. On the benchmark workload, this added massive I/O overhead.

### The Compound Effects

Beyond the two critical bugs, the analysis found a pattern of **individually reasonable but collectively catastrophic** design choices:

- **AST cloning**: The query planner cloned the entire AST for every operation instead of using references
- **4KB heap allocations**: Every expression evaluation allocated 4KB on the heap, even for simple integer comparisons
- **Schema reload on every query**: The schema was parsed from scratch for each statement instead of being cached
- **Eager string formatting**: Error messages were constructed even when no error occurred

Each of these is the kind of choice an LLM makes because it produces *correct* code — code that compiles, runs, and gives the right answer. But a human database engineer would never make these choices because they understand *performance implications*, not just *correctness*.

### The Second Case Study: The Disk Cleanup Daemon

The same analysis included a second example: an LLM asked to "write a disk cleanup daemon" produced **82,000 lines of Rust** — a full daemon with configuration files, logging, plugin architecture, health checks, metrics, and graceful shutdown.

The equivalent functionality? A one-line cron job: `0 * * * * find /tmp -mtime +7 -delete`

This illustrates a different failure mode: LLMs don't ask "is this the right approach?" They implement whatever you ask for, at whatever scale seems appropriate, without questioning whether the task warrants the complexity.

### The Sycophancy Problem

The analysis connected these failures to a deeper issue in LLM behavior: **sycophancy**. LLMs trained with RLHF (reinforcement learning from human feedback) are optimized to produce outputs that humans rate favorably. This creates a bias toward:

- **Agreement over correctness**: The model implements what you ask rather than questioning whether you should ask it
- **Plausible structure over deep understanding**: Code that *looks* like it was written by an expert (proper error handling, clean abstractions, good naming) but misses the domain-specific optimizations that actually matter
- **Completeness over minimalism**: Rather than suggesting a simpler approach, the model builds the full system you described

As the analysis concludes: *"The code is not yours until you understand it well enough to break it."*

### What It Teaches

The SQLite case study is the definitive example of **plausible vs. correct code**:

1. **Plausible code compiles and runs.** The 576,000-line Rust implementation compiled, executed queries, and returned correct results. By many metrics, it "worked."
2. **Correct code requires domain knowledge.** The iPK optimization, fdatasync batching, and schema caching are not things you learn from code patterns — they come from understanding how databases actually work.
3. **Benchmarks reveal what tests hide.** A test suite that checks "does this query return the right rows?" would pass. Only performance benchmarking revealed the 20,171x slowdown.
4. **LLMs optimize for appearance, not performance.** The code had clean error handling, proper Rust idioms, and good structure. It just missed every optimization that matters.
5. **The right question is not "does it work?" but "does it work well enough?"** For a toy project, 20,171x slower might not matter. For production database software, it is catastrophic.

## Case Study 5: Scimax for VS Code — 176,000 Lines in 43 Days

### The Project

[Scimax for VS Code](https://github.com/jkitchin/scimax_vscode) is a VS Code extension that brings the scientific computing environment of [Scimax](https://github.com/jkitchin/scimax) (an Emacs starter kit for scientists and engineers) to VS Code. The README states explicitly: **"This project is 100% vibe-engineered with Claude Code."**

The project was built by Professor John Kitchin, who has maintained the Emacs version of Scimax for over a decade. He used Claude Code to reimplement the entire system as a TypeScript VS Code extension.

### The Numbers

| Metric | Value |
|--------|-------|
| First commit | January 11, 2026 |
| Most recent commit (at time of analysis) | February 23, 2026 |
| Total elapsed time | 43 calendar days |
| Active development days | 28 |
| Total commits | 549 |
| Average commits per active day | ~20 |
| Peak commits in one day | 57 (January 24) |
| Source code (non-test TypeScript) | ~138,000 lines across 234 files |
| Test code | ~37,600 lines across 70 files |
| Total TypeScript | ~176,000 lines |
| AI co-authored commits | 319 of 549 (58%) |
| AI model used | Claude Opus 4.5 (314 commits), Claude Opus 4.6 (5 commits) |

### The Development Story

The commit history reveals three distinct phases:

**Phase 1 — Explosive Build (Jan 11-24): 469 commits in 14 days**

Day 1 (January 11) saw 33 commits taking the project from zero to a working extension with journal support, citation management, database search with FTS5 and vector search, org-mode syntax highlighting, and fuzzy search. By day 3, the project had a complete org-mode parser, four export backends (HTML, LaTeX, DOCX, Jupyter), Jupyter kernel integration via ZeroMQ, and was at v0.2.0.

The following days added LaTeX equation preview, SyncTeX bidirectional sync, Babel code execution, CI pipeline, Zotero integration, encryption, a snippet system, a help system, a publishing system with book theme, DOCX export with math support, and advanced spreadsheet formulas.

**Phase 2 — Feature Completion (Jan 25 - Feb 4): 65 commits**

A Dired-style file manager, refile functionality, SOTA search with reranking, link graph visualization, a plugin/adapter architecture, and the v0.3.0 release.

**Phase 3 — Stabilization (Feb 11-23): 5 commits**

Rectangle editing, LaTeX export fixes, and v0.3.1.

### Why It Worked

Several factors made this project unusually successful:

1. **Deep domain expertise.** Kitchin maintained the Emacs version of Scimax for over a decade. He knew exactly what the software needed to do, what the edge cases were, and how users interact with it. The AI provided implementation speed; the human provided design knowledge.

2. **A living specification.** The existing Emacs Scimax served as a detailed specification. Every feature had a reference implementation that defined expected behavior.

3. **A comprehensive CLAUDE.md.** The project includes a 24KB `CLAUDE.md` file with detailed build commands, architecture documentation, and coding guidelines — providing persistent context across sessions.

4. **Tests alongside features.** 70 test files (37,600 lines of test code) were created alongside the source, providing ongoing verification.

5. **Rapid iteration.** The burst pattern (~20 commits/day) suggests a tight feedback loop: prompt, review, test, commit, repeat.

### What It Teaches

This case study illustrates the **expert-AI collaboration pattern**: a human with deep domain knowledge directing an AI that provides implementation speed. Key observations:

- **Domain expertise is the bottleneck, not coding speed.** Kitchin's decade of Scimax experience made the feature decisions. The AI accelerated implementation, but the human provided the irreplaceable judgment about what to build and how it should work.
- **An existing system is the best specification.** Porting a mature system (like Willison's JustHTML case) provides a clear target. The AI knows when it is done because the reference implementation defines "done."
- **The pace is not sustainable indefinitely.** Phase 1 averaged 33 commits/day. Phase 3 averaged less than 1. The burst pattern suggests intense cognitive load during active development, consistent with the burnout risks discussed in the AI Human Factors module.
- **176,000 lines in 43 days is remarkable** — but only because a human knew exactly what those lines needed to do.

## Cross-Case Analysis: What Predicts Success?

Looking across all five cases, clear patterns emerge:

### The Success Formula

| Factor | JustHTML | CCC Compiler | FastRender | SQLite Rewrite | Scimax VS Code |
|--------|---------|-------------|------------|----------------|----------------|
| Comprehensive test suite | 9,200 tests | GCC torture tests | Failing CI | None (benchmarks revealed bugs) | 37,600 lines of tests |
| Clear specification | Python source | C standard + GCC oracle | "Build a browser" | SQLite behavior (not internals) | Emacs Scimax |
| Bounded scope | Port one library | Implement C spec | Entire browser engine | Rewrite a database engine | Port known system |
| Domain expertise | Willison (experienced) | Carlini (researcher) | GPT-5.2 agents | LLM only | Kitchin (decade of Scimax) |
| Human role | Specification + setup | Orchestration + infra | Minimal | Minimal | Direction + review |
| Outcome | 99.9% tests passing | 99% tests, limited usability | Doesn't compile | Compiles, 20,171x slower | Working extension |

**The pattern is clear:** AI-assisted development succeeds when there is (1) a test suite for verification, (2) a clear specification of what to build, (3) bounded scope, and (4) human expertise guiding the process. Remove any of these and quality degrades rapidly.

### The Three Roles Humans Play

Across all cases, humans served three irreplaceable functions:

1. **Specification**: Deciding *what* to build (the plan in plan-execute)
2. **Verification infrastructure**: Creating or selecting the test suite that tells AI when it is wrong
3. **Quality judgment**: Evaluating whether passing tests actually means the software is good (Lattner's "optimizing for tests vs. building abstractions" distinction)

AI handled the implementation, but every successful case had a human providing the frame within which the AI operated.

### Lines of Code Is Not a Metric

| Project | Lines of Code | Quality |
|---------|--------------|--------|
| JustHTML | 9,000 | Excellent (99.9% tests passing) |
| CCC Compiler | 100,000 | Good (99% tests, limited usability) |
| Scimax VS Code | 176,000 | Good (working extension, 37K test lines) |
| SQLite Rewrite | 576,000 | Poor (compiles, 20,171x slower) |
| FastRender | 3,000,000 | Poor (doesn't compile) |

FastRender produced 333x more code than JustHTML and was 333x worse. The SQLite rewrite produced plausible, compiling code — but missed every optimization that matters. AI can produce code at arbitrary scale — the question is whether that code is *correct* and *performant*.

### The Test Suite Is the Ceiling

In every successful case, the test suite determined the maximum achievable quality:

- **JustHTML**: 9,200 external tests → 99.9% passing
- **CCC**: GCC torture tests → 99% passing, but tests measured *compilation correctness*, not *usability* — so the compiler passes tests but can't compile "Hello World" without manual library paths
- **Scimax VS Code**: 37,600 lines of test code → working extension
- **SQLite Rewrite**: no performance tests → 20,171x slower (correctness tests would have passed)
- **FastRender**: no passing tests → doesn't compile

If your tests measure the wrong thing (or nothing), AI will optimize for the wrong thing (or nothing). The SQLite case adds an important nuance: **correctness tests are necessary but not sufficient**. The rewrite returned correct query results — but performance benchmarks revealed it was unusable. Your test suite must measure what you actually care about.

### Plausible vs. Correct Code

The SQLite case introduces a critical distinction that runs through all five cases:

- **JustHTML**: Plausible *and* correct — verified by 9,200 tests
- **CCC Compiler**: Plausible and mostly correct — but "correct" was defined narrowly by the test suite
- **FastRender**: Not even plausible — doesn't compile
- **SQLite Rewrite**: Plausible but *incorrect* — compiles, runs, returns right answers, but 20,171x slower
- **Scimax VS Code**: Plausible and correct — verified by tests and domain expert review

LLMs are trained to produce plausible output. Plausible code looks right: proper error handling, clean abstractions, good naming conventions. But plausibility is about *appearance*, while correctness is about *behavior*. The gap between them is where domain expertise lives.

### Exercise 1: Evaluate a Claim (15 min)

Find an AI coding claim on social media, a blog post, or a news article (e.g., "AI built X in Y hours"). Apply the success factors from the cross-case analysis:

1. Was there a comprehensive test suite? How do you know?
2. Was there a clear specification, or was the scope open-ended?
3. What was the human's role? Specification? Orchestration? Minimal?
4. Can you find independent verification (community reaction, code review, CI status)?
5. How many lines of code were produced? Is that a sign of quality or bloat?

Write a brief evaluation (1 paragraph) of whether the claim is credible.

*Your evaluation here.*

### Exercise 2: Design an AI-Friendly Project (10 min)

Based on the patterns from these case studies, design a small project (scope: one weekend) that would be well-suited for AI-assisted development. Your design should specify:

1. **What to build** (clear specification)
2. **How to verify it** (test suite or reference implementation)
3. **What the human provides** (domain knowledge, architectural decisions)
4. **What the AI provides** (implementation, boilerplate, translation)
5. **What could go wrong** (where might AI produce plausible but wrong output?)

Now design a project that would be poorly suited for AI. What makes it different?

*Your project designs here.*

### Exercise 3: Repo Archaeology (15 min)

Pick any open-source project that claims AI involvement (check the README or commit messages for "Co-Authored-By" tags, mentions of Claude/Copilot/GPT, or "vibe coded" declarations). Using git commands, answer:

1. How many commits are AI co-authored vs human-only? (`git log --grep="Co-Authored-By" | wc -l`)
2. What was the development timeline? (`git log --format="%ai" | sort | head -1` for first commit)
3. What is the commit velocity pattern? (bursts or steady?)
4. Is there a test suite? What is the ratio of test code to source code?
5. Does the CI pass? Check the GitHub Actions tab.

What does the git history tell you about how AI was used in the project?

*Your analysis here.*

## Tips and Tricks

- **Always check the repo before believing the headline.** Does it compile? Does CI pass? What do the issues say? Social media posts are marketing; the code is the truth.
- **Lines of code measures effort, not value.** 3 million lines of code that doesn't compile is worse than 100 lines that work perfectly.
- **Look for the test suite first.** In every successful AI project, the test suite was the critical enabler. If there is no test suite, there is no quality guarantee.
- **Porting and translation are AI's sweet spot.** When the *what* is already decided (by an existing implementation or specification), AI can focus on the *how* and produce excellent results.
- **Open-ended design is AI's weak spot.** "Build a browser" or "design a novel algorithm" requires judgment that AI does not have. Bound the scope before engaging AI.
- **Expert direction + AI implementation is the winning pattern.** The human provides domain knowledge, specification, and quality judgment. The AI provides speed.
- **Watch the commit velocity pattern.** Bursts of 50+ commits/day followed by near-silence often indicate unsustainable pace. Steady, moderate velocity suggests healthier development.
- **"Vibe-coded" is not a pejorative — it is information.** It tells you the project was AI-assisted and you should evaluate it accordingly: check tests, check CI, check whether someone understands the code.

## Foundational Concepts to Commit to Memory

- **AI-assisted development succeeds when there is a test suite, a clear specification, bounded scope, and human expertise.** Remove any of these and quality degrades rapidly.
- **The test suite is the ceiling for AI code quality.** AI optimizes for whatever signal it has. If the signal is a comprehensive test suite, it produces good code. If there are no tests, it produces plausible-looking code of unknown quality.
- **Correctness tests are necessary but not sufficient.** The SQLite rewrite passed correctness tests (right query results) but was 20,171x slower. Your tests must measure what you actually care about — including performance, usability, and edge cases.
- **Plausible code is not correct code.** LLMs produce code that *looks* right: clean abstractions, good naming, proper error handling. But plausibility is about appearance; correctness is about behavior. The gap between them is where domain expertise lives.
- **Lines of code is not a metric of success.** The best AI-assisted projects (JustHTML) produced the least code. The worst (FastRender) produced the most.
- **Translation and porting are AI's strongest use cases.** When an existing implementation defines the target behavior, AI can focus on mechanical translation rather than design.
- **The human role shifts from implementation to orchestration.** In every successful case, the human decided *what* to build and *how to verify it*. The AI decided *how to implement it*.
- **"Passing tests" and "good software" are not the same thing.** The CCC compiler passed 99% of tests but couldn't compile Hello World without manual configuration. Tests measure what they measure, not what you wish they measured.
- **LLMs are sycophantic by design.** They implement what you ask rather than questioning whether you should ask it. They build 82,000-line daemons instead of suggesting a one-line cron job. Always ask: is this the simplest solution?
- **Always verify AI project claims independently.** Check the repository: does it compile? Does CI pass? What do community reviewers say? What do benchmarks show? The gap between announcement and reality can be enormous.

## Knowing vs. Doing: Reflection

The five projects in this module were all built with AI, but their outcomes could not be more different. JustHTML and Scimax VS Code are genuinely useful software. The CCC compiler is an impressive technical achievement with important limitations. The SQLite rewrite compiled and ran but was 20,000 times too slow. FastRender is a cautionary tale about scale without quality.

What separates them is not AI capability — all five used frontier models. It is the *human contribution*: the specification, the test suite, the domain expertise, and the judgment about what "done" means. Willison's 8 prompts and comprehensive test suite produced better software than hundreds of GPT-5.2 agents running for a week without quality control. Kitchin's decade of domain knowledge turned 176,000 lines of AI-generated code into a working product. The SQLite rewrite had no such human guidance — and the LLM produced code that *looked* expert but missed every optimization an actual database engineer would know.

The SQLite case teaches perhaps the most important lesson: **code that compiles and returns correct results can still be fundamentally broken**. The 20,171x slowdown was invisible to any test that checked "does this query return the right rows?" Only benchmarking — a form of verification that goes beyond correctness to measure *quality* — revealed the problem. This is why domain expertise matters: an expert knows not just *what* the code should do, but *how well* it should do it.

This has direct implications for your work. When you start a project with AI assistance, the quality of your outcome will be determined by three things you do *before* the AI writes a single line of code: (1) how clearly you specify what you want, (2) how you plan to verify the output — including performance, not just correctness, and (3) how much domain knowledge you bring to the evaluation. The AI provides the speed. You provide the direction.

As you build your course project and enter your career, you will encounter many AI coding claims. Some will be genuine breakthroughs. Some will be FastRender. Some will be SQLite rewrites — impressive on the surface, broken underneath. The skill you need is not how to prompt an AI — it is how to evaluate the result. Check the repo. Run the tests. Run the benchmarks. Read the community reaction. The code is the truth.

## References

### Case Study 1: JustHTML Port
- Willison, S. (2025). [Porting JustHTML from Python to JavaScript using Codex CLI](https://simonwillison.net/2025/Dec/15/porting-justhtml/). Simon Willison's Weblog.
- [simonw/justjshtml](https://github.com/simonw/justjshtml) — GitHub repository for the ported library.

### Case Study 2: Claude's C Compiler
- Carlini, N. (2026). [Building a C Compiler with a Team of Parallel Claudes](https://www.anthropic.com/engineering/building-c-compiler). Anthropic Engineering Blog.
- [anthropics/claudes-c-compiler](https://github.com/anthropics/claudes-c-compiler) — GitHub repository.
- Willison, S. (2026). [The Claude C Compiler: What It Reveals About the Future of Software](https://simonwillison.net/2026/Feb/22/ccc/). Simon Willison's Weblog (includes Chris Lattner's analysis).
- [Anthropic's AI-built C compiler is not all that impressive](https://www.theregister.com/2026/02/13/anthropic_c_compiler/). The Register.
- [Sixteen Claude Agents Built a C Compiler without Human Intervention... Almost](https://www.infoq.com/news/2026/02/claude-built-c-compiler/). InfoQ.

### Case Study 3: Cursor's FastRender Browser
- [Cursor shows AI agents capable of shoddy code at scale](https://www.theregister.com/2026/01/22/cursor_ai_wrote_a_browser/). The Register.
- [Cursor CEO Built a Browser using AI, but Does It Really Work?](https://www.finalroundai.com/blog/cursor-ceo-browser-made-using-ai). Final Round AI.
- [Cursor's Reputation Plummets Overnight: AI's Claim of Writing 3-Million-Line Browser Code Exposed](https://eu.36kr.com/en/p/3643187094507394). 36Kr.
- [When AI 'builds a browser,' check the repo before believing the hype](https://news.ycombinator.com/item?id=46769965). Hacker News discussion.
- Software Improvement Group. [Quality of FastRender](https://www.softwareimprovementgroup.com/blog/quality-of-fastrender/). SIG Blog (professional code quality analysis: 1.3/5 maintainability).

### Case Study 4: LLM SQLite Rewrite
- Hōrōshi / KatanaQuant. [Your LLM Doesn't Write Correct Code. It Writes Plausible Code](https://katanaquant.com/blog/llm-plausible-code). KatanaQuant Blog.
- Hoare, C. A. R. (1981). The Emperor's Old Clothes. *Communications of the ACM*, 24(2), 75–83.
- Skiena, S. S. (2020). *The Algorithm Design Manual* (3rd ed.). Springer.

### Case Study 5: Scimax for VS Code
- [jkitchin/scimax_vscode](https://github.com/jkitchin/scimax_vscode) — GitHub repository.
- [jkitchin/scimax](https://github.com/jkitchin/scimax) — the original Emacs Scimax project.